<div align="center">
  <img src="https://raw.githubusercontent.com/WSU-AI-in-ME/ai-in-me-1/main/img/wsu_logo_horizontal.png" alt="Wayne State University Logo" width="320">
  <h1>Practice 5: Classification and Model Performance Evaluation</h1>
</div>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/WSU-AI-in-ME/ai-in-me-1/blob/main/practice/week05/practice05_classification_experiments.ipynb)

[Course Repository](https://github.com/WSU-AI-in-ME/ai-in-me-1) · [Practice Index](https://github.com/WSU-AI-in-ME/ai-in-me-1/blob/main/Practice_index.ipynb)

### ME 5995 — AI in Mechanical Engineering I: Fundamentals of Manufacturing Data Science
**Wayne State University · Optional, ungraded self-study · No submission**

**Version: Student starter**

Try selected exercises before consulting the [paired notebook](practice05_classification_experiments_solution.ipynb).
Open in Colab and save a personal copy before editing.

## Student Information

Optional for your personal copy. Double-click to edit.

**Student name:** TODO: Enter your name  
**WSU AccessID:** TODO: Enter your WSU AccessID

## Overview

Explore classification errors, kNN and Gaussian Naive Bayes by changing one setting
at a time. This is an optional exercise bank, not an assignment to finish in full.
Prerequisites: Lab 5's fit/predict, training/validation and confusion matrices.

**Recommended route: A1 → A2 → B1 → B3 → C2 → D2.** Before C2, run C's one-feature
warm-up. The other ten exercises are extensions; no derivations are required.

| Module | Focus |
|---|---|
| [A — Metrics](#module-a) | A1–A4: counts, imbalance, thresholds, macro-F1 |
| [B — kNN](#module-b) | B1–B4: neighbors, voting, scaling, label noise |
| [C — GaussianNB](#module-c) | Warm-up → C2 priors; C3 duplication; C1 smoothing |
| [D — PHM](#module-d) | D1–D3: binary kNN, multiclass comparison, threshold |
| [E — Optional LDA/QDA](#module-e) | E1–E2: boundaries and training sample size |

### How to use each exercise

1. Run its module setup. Each module is independent; earlier exercise answers are not needed.
2. Read the question and predict a change. Run the completed example, edit the YOUR TURN cell,
   then run its comparison and plot cells. Reporting code is supplied; do not reconstruct it.
3. Copy results into the record table before trying another setting, then answer the specific
   question in short phrases. Tables are edited manually; unchanged results are valid findings.
4. When changing a setting, rerun its action and reporting/plot cells. When switching modules,
   rerun the new setup because variable names are reused. Consult the paired solution after trying.

An unchanged starter is runnable but has not performed the requested experiment.
Record results once in the table; your response can refer to them instead of repeating numbers.

### Setup

Colab: run the imports. Locally: install NumPy, pandas, Matplotlib, scikit-learn and
Jupyter in your Python environment. A/B/C/E use synthetic data. D loads the course
PHM CSV online, or uses a copy placed beside the notebook. No raw signals are needed.
Held-out rows are validation, not final test data; repeated comparisons can overfit them.

Planning estimates: 20–30 minutes per module; 120–150 for the full bank. Actual times
are unmeasured. There is no submission and no need to complete the full bank.

<a id="module-a"></a>

## A — Read errors before trusting a score

All labels in this module are illustrative. Class 1 is the positive class.
Precision = TP/(TP+FP); recall = TP/(TP+FN); F1 = 2TP/(2TP+FP+FN).
The confusion matrix uses actual rows and predicted columns. Run A setup first.

These arrays are invented predictions for learning how metrics work. Changing an entry here is not training a classifier. Never repair real validation predictions by copying their known correct labels.

In [ ]:
# PROVIDED SETUP — imports make tools available; they do not train a model.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.datasets import make_moons, make_classification
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
plt.rcParams['font.size'] = 11
plt.rcParams['figure.dpi'] = 100

### A1 — Recommended: Find the missed positive

Change the final prediction from 0 to 1. Which matrix entry moves?

In [ ]:
# COMPLETED EXAMPLE — run unchanged.
actual = np.array([0, 0, 0, 1, 1, 1])
reference_pred = np.array([0, 0, 1, 1, 1, 0])

In [ ]:
# YOUR TURN — edit the marked setting/expression.
# TODO: Correct only the final prediction.
changed_pred = np.array([0, 0, 1, 1, 1, 0])

In [ ]:
# PROVIDED REPORTING — optional reading; run unchanged.
# compare the two predictions on the SAME validation labels.
# Class 1 is positive. zero_division=0 reports zero for an undefined metric.
reference_accuracy = accuracy_score(actual, reference_pred)
changed_accuracy = accuracy_score(actual, changed_pred)
reference_precision = precision_score(actual, reference_pred, zero_division=0)
changed_precision = precision_score(actual, changed_pred, zero_division=0)
reference_recall = recall_score(actual, reference_pred, zero_division=0)
changed_recall = recall_score(actual, changed_pred, zero_division=0)
# Compare actual labels with predictions. Binary F1 uses class 1; macro-F1 averages class F1 values equally.
reference_f1 = f1_score(actual, reference_pred, zero_division=0)
# Compare actual labels with predictions. Binary F1 uses class 1; macro-F1 averages class F1 values equally.
changed_f1 = f1_score(actual, changed_pred, zero_division=0)
comparison = pd.DataFrame()
comparison['Case'] = ['Completed example', 'Your variant']
comparison['Accuracy'] = [reference_accuracy, changed_accuracy]
comparison['Precision (class 1)'] = [reference_precision, changed_precision]
comparison['Recall (class 1)'] = [reference_recall, changed_recall]
comparison['F1 (class 1)'] = [reference_f1, changed_f1]
# Count errors explicitly so you do not need a matrix for every experiment.
reference_matrix = confusion_matrix(actual, reference_pred, labels=[0, 1])
# Count actual labels by row and predicted labels by column, in the specified label order.
changed_matrix = confusion_matrix(actual, changed_pred, labels=[0, 1])
comparison['False positives'] = [reference_matrix[0, 1], changed_matrix[0, 1]]
comparison['False negatives'] = [reference_matrix[1, 0], changed_matrix[1, 0]]
# Round only the displayed table; model selection still uses the unrounded scores.
display(comparison.round(3))
changed_prediction_count = np.count_nonzero(reference_pred != changed_pred)
print('Predictions that changed:', changed_prediction_count)
labels = [0, 1]
names = ['Class 0', 'Class 1']
# fig is the full figure; axes[0] and axes[1] are its two panels.
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
# Rows are actual classes and columns are predictions. Keep the same color scale.
ConfusionMatrixDisplay.from_predictions(actual, reference_pred, labels=labels,
    display_labels=names, ax=axes[0], colorbar=False, cmap='Blues', xticks_rotation=20)
ConfusionMatrixDisplay.from_predictions(actual, changed_pred, labels=labels,
    display_labels=names, ax=axes[1], colorbar=False, cmap='Blues', xticks_rotation=20)
axes[0].images[0].set_clim(0, len(actual))
axes[1].images[0].set_clim(0, len(actual))
axes[0].set_title('Completed example')
axes[1].set_title('Your variant')
fig.tight_layout()  # Make room for labels.
plt.show()  # Display the figure.

#### Record your results

| Setting | Positive-class F1 | FP / FN | Brief observation |
|---|---|---|---|
| Completed example | ___ | ___ | ___ |
| My variant: ___ | ___ | ___ | ___ |

**Your observation:** Which count changes from FN to TP? Calculate recall before and after. Why is editing this illustrative prediction not a way to improve a trained model?

TODO: Write your response using the output.

### A2 — Recommended: High accuracy, no detection

The sample has 18 negatives and 2 positives. Try detecting one positive while adding one false alarm. Compare accuracy and F1.

In [ ]:
# COMPLETED EXAMPLE — run unchanged.
actual = np.array([0] * 18 + [1, 1])
reference_pred = np.zeros(20, dtype=int)  # Always predict class 0.

In [ ]:
# YOUR TURN — edit the marked setting/expression.
changed_pred = reference_pred.copy()  # A separate array; keep the reference unchanged.
# TODO: Set positions 0 and 18 to 1.
changed_pred[0] = 0
changed_pred[18] = 0

In [ ]:
# PROVIDED REPORTING — optional reading; run unchanged.
# compare the two predictions on the SAME validation labels.
# Class 1 is positive. zero_division=0 reports zero for an undefined metric.
reference_accuracy = accuracy_score(actual, reference_pred)
changed_accuracy = accuracy_score(actual, changed_pred)
reference_precision = precision_score(actual, reference_pred, zero_division=0)
changed_precision = precision_score(actual, changed_pred, zero_division=0)
reference_recall = recall_score(actual, reference_pred, zero_division=0)
changed_recall = recall_score(actual, changed_pred, zero_division=0)
# Compare actual labels with predictions. Binary F1 uses class 1; macro-F1 averages class F1 values equally.
reference_f1 = f1_score(actual, reference_pred, zero_division=0)
# Compare actual labels with predictions. Binary F1 uses class 1; macro-F1 averages class F1 values equally.
changed_f1 = f1_score(actual, changed_pred, zero_division=0)
comparison = pd.DataFrame()
comparison['Case'] = ['Completed example', 'Your variant']
comparison['Accuracy'] = [reference_accuracy, changed_accuracy]
comparison['Precision (class 1)'] = [reference_precision, changed_precision]
comparison['Recall (class 1)'] = [reference_recall, changed_recall]
comparison['F1 (class 1)'] = [reference_f1, changed_f1]
# Count errors explicitly so you do not need a matrix for every experiment.
reference_matrix = confusion_matrix(actual, reference_pred, labels=[0, 1])
# Count actual labels by row and predicted labels by column, in the specified label order.
changed_matrix = confusion_matrix(actual, changed_pred, labels=[0, 1])
comparison['False positives'] = [reference_matrix[0, 1], changed_matrix[0, 1]]
comparison['False negatives'] = [reference_matrix[1, 0], changed_matrix[1, 0]]
# Round only the displayed table; model selection still uses the unrounded scores.
display(comparison.round(3))
changed_prediction_count = np.count_nonzero(reference_pred != changed_pred)
print('Predictions that changed:', changed_prediction_count)
labels = [0, 1]
names = ['Class 0', 'Class 1']
# fig is the full figure; axes[0] and axes[1] are its two panels.
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
# Rows are actual classes and columns are predictions. Keep the same color scale.
ConfusionMatrixDisplay.from_predictions(actual, reference_pred, labels=labels,
    display_labels=names, ax=axes[0], colorbar=False, cmap='Blues', xticks_rotation=20)
ConfusionMatrixDisplay.from_predictions(actual, changed_pred, labels=labels,
    display_labels=names, ax=axes[1], colorbar=False, cmap='Blues', xticks_rotation=20)
axes[0].images[0].set_clim(0, len(actual))
axes[1].images[0].set_clim(0, len(actual))
axes[0].set_title('Completed example')
axes[1].set_title('Your variant')
fig.tight_layout()  # Make room for labels.
plt.show()  # Display the figure.

#### Record your results

| Setting | Positive-class F1 | FP / FN | Brief observation |
|---|---|---|---|
| Completed example | ___ | ___ | ___ |
| My variant: ___ | ___ | ___ | ___ |

**Your observation:** Record both accuracies and positive F1 scores. Which prediction set detects any positives, and why does accuracy fail to distinguish the cases?

TODO: Write your response using the output.

### A3 — Additional: The threshold trade-off

Keep these illustrative probabilities fixed. Lower the decision threshold from 0.50 to 0.30, then inspect FP/FN.

In [ ]:
# COMPLETED EXAMPLE — run unchanged.
actual = np.array([0, 0, 0, 0, 1, 1, 1, 1])
probability = np.array([0.10, 0.20, 0.35, 0.60, 0.25, 0.45, 0.70, 0.90])
# The comparison is True where the estimated positive-class probability reaches the cutoff.
reference_pred = (probability >= 0.50).astype(int)

In [ ]:
# YOUR TURN — edit the marked setting/expression.
threshold = 0.50  # TODO: Try 0.30.
# The comparison is True where the estimated positive-class probability reaches the cutoff.
is_positive = probability >= threshold
changed_pred = is_positive.astype(int)  # True -> 1; False -> 0.

In [ ]:
# PROVIDED REPORTING — optional reading; run unchanged.
# compare the two predictions on the SAME validation labels.
# Class 1 is positive. zero_division=0 reports zero for an undefined metric.
reference_accuracy = accuracy_score(actual, reference_pred)
changed_accuracy = accuracy_score(actual, changed_pred)
reference_precision = precision_score(actual, reference_pred, zero_division=0)
changed_precision = precision_score(actual, changed_pred, zero_division=0)
reference_recall = recall_score(actual, reference_pred, zero_division=0)
changed_recall = recall_score(actual, changed_pred, zero_division=0)
# Compare actual labels with predictions. Binary F1 uses class 1; macro-F1 averages class F1 values equally.
reference_f1 = f1_score(actual, reference_pred, zero_division=0)
# Compare actual labels with predictions. Binary F1 uses class 1; macro-F1 averages class F1 values equally.
changed_f1 = f1_score(actual, changed_pred, zero_division=0)
comparison = pd.DataFrame()
comparison['Case'] = ['Completed example', 'Your variant']
comparison['Accuracy'] = [reference_accuracy, changed_accuracy]
comparison['Precision (class 1)'] = [reference_precision, changed_precision]
comparison['Recall (class 1)'] = [reference_recall, changed_recall]
comparison['F1 (class 1)'] = [reference_f1, changed_f1]
# Count errors explicitly so you do not need a matrix for every experiment.
reference_matrix = confusion_matrix(actual, reference_pred, labels=[0, 1])
# Count actual labels by row and predicted labels by column, in the specified label order.
changed_matrix = confusion_matrix(actual, changed_pred, labels=[0, 1])
comparison['False positives'] = [reference_matrix[0, 1], changed_matrix[0, 1]]
comparison['False negatives'] = [reference_matrix[1, 0], changed_matrix[1, 0]]
# Round only the displayed table; model selection still uses the unrounded scores.
display(comparison.round(3))
changed_prediction_count = np.count_nonzero(reference_pred != changed_pred)
print('Predictions that changed:', changed_prediction_count)
labels = [0, 1]
names = ['Class 0', 'Class 1']

#### Record your results

| Setting | Positive-class F1 | FP / FN | Brief observation |
|---|---|---|---|
| Completed example | ___ | ___ | ___ |
| My variant: ___ | ___ | ___ | ___ |

**Your observation:** Record FP and FN at each threshold. Which probability values change their predicted class, and how do those cases explain the trade-off?

TODO: Write your response using the output.

### A4 — Additional: Macro-F1 notices the rare classes

Predict one class-1 and one class-2 example correctly instead of always predicting class 0. Compare macro and weighted F1.

Support is the number of actual examples in a class. Macro-F1 averages the class F1 scores equally; weighted-F1 weights them by support.

In [ ]:
# COMPLETED EXAMPLE — run unchanged.
actual = np.array([0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 2, 2])
reference_pred = np.zeros(12, dtype=int)

In [ ]:
# YOUR TURN — edit the marked setting/expression.
changed_pred = reference_pred.copy()
# TODO: Set position 8 to 1 and position 10 to 2.
changed_pred[8] = 0
changed_pred[10] = 0

In [ ]:
# PROVIDED REPORTING — optional reading; run unchanged.
# macro-F1 weights each class equally; weighted-F1 uses support.
reference_f1 = f1_score(actual, reference_pred, labels=[0, 1, 2], average='macro', zero_division=0)
# Compare actual labels with predictions. Binary F1 uses class 1; macro-F1 averages class F1 values equally.
changed_f1 = f1_score(actual, changed_pred, labels=[0, 1, 2], average='macro', zero_division=0)
# Compare actual labels with predictions. Binary F1 uses class 1; macro-F1 averages class F1 values equally.
reference_weighted = f1_score(actual, reference_pred, labels=[0, 1, 2], average='weighted', zero_division=0)
# Compare actual labels with predictions. Binary F1 uses class 1; macro-F1 averages class F1 values equally.
changed_weighted = f1_score(actual, changed_pred, labels=[0, 1, 2], average='weighted', zero_division=0)
comparison = pd.DataFrame()
comparison['Case'] = ['Completed example', 'Your variant']
comparison['Macro-F1'] = [reference_f1, changed_f1]
comparison['Weighted-F1'] = [reference_weighted, changed_weighted]
# Round only the displayed table; model selection still uses the unrounded scores.
display(comparison.round(3))
labels = [0, 1, 2]
names = ['Class 0', 'Class 1', 'Class 2']
print(classification_report(actual, changed_pred, labels=labels, digits=3, zero_division=0))
# fig is the full figure; axes[0] and axes[1] are its two panels.
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
# Rows are actual classes and columns are predictions. Keep the same color scale.
ConfusionMatrixDisplay.from_predictions(actual, reference_pred, labels=labels,
    display_labels=names, ax=axes[0], colorbar=False, cmap='Blues', xticks_rotation=20)
ConfusionMatrixDisplay.from_predictions(actual, changed_pred, labels=labels,
    display_labels=names, ax=axes[1], colorbar=False, cmap='Blues', xticks_rotation=20)
axes[0].images[0].set_clim(0, len(actual))
axes[1].images[0].set_clim(0, len(actual))
axes[0].set_title('Completed example')
axes[1].set_title('Your variant')
fig.tight_layout()  # Make room for labels.
plt.show()  # Display the figure.

#### Record your results

| Setting | Macro-F1 | Class error to inspect | Brief observation |
|---|---|---|---|
| Completed example | ___ | ___ | ___ |
| My variant: ___ | ___ | ___ | ___ |

**Your observation:** Record macro-F1 and weighted-F1 before and after. Which class has the largest support, and why do the two averages differ?

TODO: Write your response using the output.

<a id="module-b"></a>

## B — k-nearest neighbors (kNN)

kNN stores training examples and classifies a new point using nearby examples.
Small k follows local detail; larger k averages over a wider neighborhood.
Distance depends on feature units. Uniform voting gives each neighbor one vote;
distance voting gives closer neighbors more influence. These are synthetic data.

In [ ]:
# PROVIDED SETUP — imports make tools available; they do not train a model.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.datasets import make_moons, make_classification
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
plt.rcParams['font.size'] = 11
plt.rcParams['figure.dpi'] = 100

# Synthetic features have no physical units and are not measured sensor data.
# Generate once with a fixed seed, then keep the same validation rows for comparisons.
X, y = make_moons(n_samples=240, noise=0.25, random_state=42)
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)
# Fit scaling only on training rows; use those same statistics for validation.
scaler = StandardScaler()
# Fit learns from the supplied training data. Keep validation/test data out of fitting.
scaler.fit(X_train)
# Reuse the training scaling parameters; do not learn new scaling from validation/test rows.
X_train_scaled = scaler.transform(X_train)
# Reuse the training scaling parameters; do not learn new scaling from validation/test rows.
X_valid_scaled = scaler.transform(X_valid)
print('Training / validation rows:', len(y_train), len(y_valid))

### B1 — Recommended: Change neighborhood size

Compare k=5 with k=1; then try k=15. Keep rows and scaling fixed. Does a larger neighborhood always improve F1?

In [ ]:
# COMPLETED EXAMPLE — run unchanged.
actual = y_valid
reference_model = KNeighborsClassifier(n_neighbors=5, weights='uniform')
reference_model.fit(X_train_scaled, y_train)  # Store training inputs and labels.
reference_pred = reference_model.predict(X_valid_scaled)  # Predict validation labels.

In [ ]:
# YOUR TURN — edit the marked setting/expression.
k = 5  # TODO: Try 1, then 15.
changed_model = KNeighborsClassifier(n_neighbors=k)
# Fit learns from the supplied training data. Keep validation/test data out of fitting.
changed_model.fit(X_train_scaled, y_train)
# Use the fitted model to assign labels to these rows; predict does not train the model.
changed_pred = changed_model.predict(X_valid_scaled)

In [ ]:
# PROVIDED REPORTING — optional reading; run unchanged.
# compare the two predictions on the SAME validation labels.
# Class 1 is positive. zero_division=0 reports zero for an undefined metric.
reference_accuracy = accuracy_score(actual, reference_pred)
changed_accuracy = accuracy_score(actual, changed_pred)
reference_precision = precision_score(actual, reference_pred, zero_division=0)
changed_precision = precision_score(actual, changed_pred, zero_division=0)
reference_recall = recall_score(actual, reference_pred, zero_division=0)
changed_recall = recall_score(actual, changed_pred, zero_division=0)
# Compare actual labels with predictions. Binary F1 uses class 1; macro-F1 averages class F1 values equally.
reference_f1 = f1_score(actual, reference_pred, zero_division=0)
# Compare actual labels with predictions. Binary F1 uses class 1; macro-F1 averages class F1 values equally.
changed_f1 = f1_score(actual, changed_pred, zero_division=0)
comparison = pd.DataFrame()
comparison['Case'] = ['Completed example', 'Your variant']
comparison['Accuracy'] = [reference_accuracy, changed_accuracy]
comparison['Precision (class 1)'] = [reference_precision, changed_precision]
comparison['Recall (class 1)'] = [reference_recall, changed_recall]
comparison['F1 (class 1)'] = [reference_f1, changed_f1]
# Count errors explicitly so you do not need a matrix for every experiment.
reference_matrix = confusion_matrix(actual, reference_pred, labels=[0, 1])
# Count actual labels by row and predicted labels by column, in the specified label order.
changed_matrix = confusion_matrix(actual, changed_pred, labels=[0, 1])
comparison['False positives'] = [reference_matrix[0, 1], changed_matrix[0, 1]]
comparison['False negatives'] = [reference_matrix[1, 0], changed_matrix[1, 0]]
# Round only the displayed table; model selection still uses the unrounded scores.
display(comparison.round(3))
changed_prediction_count = np.count_nonzero(reference_pred != changed_pred)
print('Predictions that changed:', changed_prediction_count)
labels = [0, 1]
names = ['Class 0', 'Class 1']
# fig is the full figure; axes[0] and axes[1] are its two panels.
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
# Rows are actual classes and columns are predictions. Keep the same color scale.
ConfusionMatrixDisplay.from_predictions(actual, reference_pred, labels=labels,
    display_labels=names, ax=axes[0], colorbar=False, cmap='Blues', xticks_rotation=20)
ConfusionMatrixDisplay.from_predictions(actual, changed_pred, labels=labels,
    display_labels=names, ax=axes[1], colorbar=False, cmap='Blues', xticks_rotation=20)
axes[0].images[0].set_clim(0, len(actual))
axes[1].images[0].set_clim(0, len(actual))
axes[0].set_title('Completed example')
axes[1].set_title('Your variant')
fig.tight_layout()  # Make room for labels.
plt.show()  # Display the figure.

In [ ]:
# PROVIDED — visualize both fitted models in the same two-feature space.
# Background colors are predictions; points show actual VALIDATION labels.
plot_train = X_train_scaled
plot_valid = X_valid_scaled
# Set plot limits from training inputs; empty regions are extrapolation.
x_axis = np.linspace(plot_train[:, 0].min() - 0.5, plot_train[:, 0].max() + 0.5, 180)
y_axis = np.linspace(plot_train[:, 1].min() - 0.5, plot_train[:, 1].max() + 0.5, 180)
grid_x, grid_y = np.meshgrid(x_axis, y_axis)  # Create a rectangular grid.
grid_inputs = np.column_stack([grid_x.ravel(), grid_y.ravel()])  # One point per row.
# Use the fitted model to assign labels to these rows; predict does not train the model.
reference_grid = reference_model.predict(grid_inputs).reshape(grid_x.shape)
# Use the fitted model to assign labels to these rows; predict does not train the model.
changed_grid = changed_model.predict(grid_inputs).reshape(grid_x.shape)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].contourf(grid_x, grid_y, reference_grid, levels=[-0.5, 0.5, 1.5], colors=['#DCEAF5', '#F7DFDF'])
axes[0].scatter(plot_valid[:, 0], plot_valid[:, 1], c=y_valid, cmap='coolwarm', edgecolors='black', s=22)
axes[0].set_xlabel('Illustrative feature 1 (z-score)')
axes[0].set_ylabel('Illustrative feature 2 (z-score)')
axes[0].set_title('Completed example')
axes[1].contourf(grid_x, grid_y, changed_grid, levels=[-0.5, 0.5, 1.5], colors=['#DCEAF5', '#F7DFDF'])
axes[1].scatter(plot_valid[:, 0], plot_valid[:, 1], c=y_valid, cmap='coolwarm', edgecolors='black', s=22)
axes[1].set_xlabel('Illustrative feature 1 (z-score)')
axes[1].set_ylabel('Illustrative feature 2 (z-score)')
axes[1].set_title('Your variant')
fig.suptitle('Synthetic data: blue = class 0; red = class 1')
fig.tight_layout()
plt.show()

#### Record your results

| Setting | Positive-class F1 | FP / FN | Brief observation |
|---|---|---|---|
| k=5 (reference) | ___ | ___ | ___ |
| k=1 | ___ | ___ | ___ |
| k=15 | ___ | ___ | ___ |

**Your observation:** Record k=1, 5 and 15. Which setting gives the highest validation F1 here? Describe one boundary change without claiming that this k is universally best.

TODO: Write your response using the output.

### B2 — Additional: Give closer neighbors more influence

Keep k=15 and compare uniform voting with distance voting.

In [ ]:
# COMPLETED EXAMPLE — run unchanged.
actual = y_valid
reference_model = KNeighborsClassifier(n_neighbors=15, weights='uniform')
reference_model.fit(X_train_scaled, y_train)  # Store training inputs and labels.
reference_pred = reference_model.predict(X_valid_scaled)  # Predict validation labels.

In [ ]:
# YOUR TURN — edit the marked setting/expression.
voting = 'uniform'  # TODO: Use 'distance'.
changed_model = KNeighborsClassifier(n_neighbors=15, weights=voting)
# Fit learns from the supplied training data. Keep validation/test data out of fitting.
changed_model.fit(X_train_scaled, y_train)
# Use the fitted model to assign labels to these rows; predict does not train the model.
changed_pred = changed_model.predict(X_valid_scaled)

In [ ]:
# PROVIDED REPORTING — optional reading; run unchanged.
# compare the two predictions on the SAME validation labels.
# Class 1 is positive. zero_division=0 reports zero for an undefined metric.
reference_accuracy = accuracy_score(actual, reference_pred)
changed_accuracy = accuracy_score(actual, changed_pred)
reference_precision = precision_score(actual, reference_pred, zero_division=0)
changed_precision = precision_score(actual, changed_pred, zero_division=0)
reference_recall = recall_score(actual, reference_pred, zero_division=0)
changed_recall = recall_score(actual, changed_pred, zero_division=0)
# Compare actual labels with predictions. Binary F1 uses class 1; macro-F1 averages class F1 values equally.
reference_f1 = f1_score(actual, reference_pred, zero_division=0)
# Compare actual labels with predictions. Binary F1 uses class 1; macro-F1 averages class F1 values equally.
changed_f1 = f1_score(actual, changed_pred, zero_division=0)
comparison = pd.DataFrame()
comparison['Case'] = ['Completed example', 'Your variant']
comparison['Accuracy'] = [reference_accuracy, changed_accuracy]
comparison['Precision (class 1)'] = [reference_precision, changed_precision]
comparison['Recall (class 1)'] = [reference_recall, changed_recall]
comparison['F1 (class 1)'] = [reference_f1, changed_f1]
# Count errors explicitly so you do not need a matrix for every experiment.
reference_matrix = confusion_matrix(actual, reference_pred, labels=[0, 1])
# Count actual labels by row and predicted labels by column, in the specified label order.
changed_matrix = confusion_matrix(actual, changed_pred, labels=[0, 1])
comparison['False positives'] = [reference_matrix[0, 1], changed_matrix[0, 1]]
comparison['False negatives'] = [reference_matrix[1, 0], changed_matrix[1, 0]]
# Round only the displayed table; model selection still uses the unrounded scores.
display(comparison.round(3))
changed_prediction_count = np.count_nonzero(reference_pred != changed_pred)
print('Predictions that changed:', changed_prediction_count)
labels = [0, 1]
names = ['Class 0', 'Class 1']

#### Record your results

| Setting | Positive-class F1 | FP / FN | Brief observation |
|---|---|---|---|
| Completed example | ___ | ___ | ___ |
| My variant: ___ | ___ | ___ | ___ |

**Your observation:** At fixed k=15, which FP/FN counts change with distance voting? Explain what closer neighbors contribute differently.

TODO: Write your response using the output.

### B3 — Recommended: An arbitrary unit change

The reference multiplies feature 1 by 100 and uses unscaled distances. Turn on training-only scaling for the same unit-changed arrays.

In [ ]:
# COMPLETED EXAMPLE — run unchanged.
actual = y_valid
train_units = X_train.copy()
valid_units = X_valid.copy()
train_units[:, 0] = train_units[:, 0] * 100  # Only the numerical unit changes.
valid_units[:, 0] = valid_units[:, 0] * 100
reference_model = KNeighborsClassifier(n_neighbors=5)
# Fit learns from the supplied training data. Keep validation/test data out of fitting.
reference_model.fit(train_units, y_train)
# Use the fitted model to assign labels to these rows; predict does not train the model.
reference_pred = reference_model.predict(valid_units)

In [ ]:
# YOUR TURN — edit the marked setting/expression.
use_scaling = False  # TODO: Change to True.
variant_train = train_units.copy()
variant_valid = valid_units.copy()
if use_scaling:
    unit_scaler = StandardScaler()
    unit_scaler.fit(train_units)  # Never learn means/SDs from validation.
    # Reuse the training scaling parameters; do not learn new scaling from validation/test rows.
    variant_train = unit_scaler.transform(train_units)
    # Reuse the training scaling parameters; do not learn new scaling from validation/test rows.
    variant_valid = unit_scaler.transform(valid_units)
changed_model = KNeighborsClassifier(n_neighbors=5)
# Fit learns from the supplied training data. Keep validation/test data out of fitting.
changed_model.fit(variant_train, y_train)
# Use the fitted model to assign labels to these rows; predict does not train the model.
changed_pred = changed_model.predict(variant_valid)

In [ ]:
# PROVIDED REPORTING — optional reading; run unchanged.
# compare the two predictions on the SAME validation labels.
# Class 1 is positive. zero_division=0 reports zero for an undefined metric.
reference_accuracy = accuracy_score(actual, reference_pred)
changed_accuracy = accuracy_score(actual, changed_pred)
reference_precision = precision_score(actual, reference_pred, zero_division=0)
changed_precision = precision_score(actual, changed_pred, zero_division=0)
reference_recall = recall_score(actual, reference_pred, zero_division=0)
changed_recall = recall_score(actual, changed_pred, zero_division=0)
# Compare actual labels with predictions. Binary F1 uses class 1; macro-F1 averages class F1 values equally.
reference_f1 = f1_score(actual, reference_pred, zero_division=0)
# Compare actual labels with predictions. Binary F1 uses class 1; macro-F1 averages class F1 values equally.
changed_f1 = f1_score(actual, changed_pred, zero_division=0)
comparison = pd.DataFrame()
comparison['Case'] = ['Completed example', 'Your variant']
comparison['Accuracy'] = [reference_accuracy, changed_accuracy]
comparison['Precision (class 1)'] = [reference_precision, changed_precision]
comparison['Recall (class 1)'] = [reference_recall, changed_recall]
comparison['F1 (class 1)'] = [reference_f1, changed_f1]
# Count errors explicitly so you do not need a matrix for every experiment.
reference_matrix = confusion_matrix(actual, reference_pred, labels=[0, 1])
# Count actual labels by row and predicted labels by column, in the specified label order.
changed_matrix = confusion_matrix(actual, changed_pred, labels=[0, 1])
comparison['False positives'] = [reference_matrix[0, 1], changed_matrix[0, 1]]
comparison['False negatives'] = [reference_matrix[1, 0], changed_matrix[1, 0]]
# Round only the displayed table; model selection still uses the unrounded scores.
display(comparison.round(3))
changed_prediction_count = np.count_nonzero(reference_pred != changed_pred)
print('Predictions that changed:', changed_prediction_count)
labels = [0, 1]
names = ['Class 0', 'Class 1']
# fig is the full figure; axes[0] and axes[1] are its two panels.
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
# Rows are actual classes and columns are predictions. Keep the same color scale.
ConfusionMatrixDisplay.from_predictions(actual, reference_pred, labels=labels,
    display_labels=names, ax=axes[0], colorbar=False, cmap='Blues', xticks_rotation=20)
ConfusionMatrixDisplay.from_predictions(actual, changed_pred, labels=labels,
    display_labels=names, ax=axes[1], colorbar=False, cmap='Blues', xticks_rotation=20)
axes[0].images[0].set_clim(0, len(actual))
axes[1].images[0].set_clim(0, len(actual))
axes[0].set_title('Completed example')
axes[1].set_title('Your variant')
fig.tight_layout()  # Make room for labels.
plt.show()  # Display the figure.

#### Record your results

| Setting | Positive-class F1 | FP / FN | Brief observation |
|---|---|---|---|
| Completed example | ___ | ___ | ___ |
| My variant: ___ | ___ | ___ | ___ |

**Your observation:** Record unscaled and scaled F1. Did multiplying a coordinate by 100 add information? Explain why distances can still change.

TODO: Write your response using the output.

### B4 — Additional: Training-label noise

Flip 20 training labels, then refit the same k=5 model. Keep validation labels unchanged. Is every error necessarily caused by the flips?

In [ ]:
# COMPLETED EXAMPLE — run unchanged.
actual = y_valid
reference_model = KNeighborsClassifier(n_neighbors=5, weights='uniform')
reference_model.fit(X_train_scaled, y_train)  # Store training inputs and labels.
reference_pred = reference_model.predict(X_valid_scaled)  # Predict validation labels.

In [ ]:
# YOUR TURN — edit the marked setting/expression.
flip_count = 0  # TODO: Try 20 (maximum 180).
noisy_labels = y_train.copy()  # Protect the original training labels.
noisy_labels[:flip_count] = 1 - noisy_labels[:flip_count]  # Swap binary 0/1.
changed_model = KNeighborsClassifier(n_neighbors=5)
# Fit learns from the supplied training data. Keep validation/test data out of fitting.
changed_model.fit(X_train_scaled, noisy_labels)
# Use the fitted model to assign labels to these rows; predict does not train the model.
changed_pred = changed_model.predict(X_valid_scaled)

In [ ]:
# PROVIDED REPORTING — optional reading; run unchanged.
# compare the two predictions on the SAME validation labels.
# Class 1 is positive. zero_division=0 reports zero for an undefined metric.
reference_accuracy = accuracy_score(actual, reference_pred)
changed_accuracy = accuracy_score(actual, changed_pred)
reference_precision = precision_score(actual, reference_pred, zero_division=0)
changed_precision = precision_score(actual, changed_pred, zero_division=0)
reference_recall = recall_score(actual, reference_pred, zero_division=0)
changed_recall = recall_score(actual, changed_pred, zero_division=0)
# Compare actual labels with predictions. Binary F1 uses class 1; macro-F1 averages class F1 values equally.
reference_f1 = f1_score(actual, reference_pred, zero_division=0)
# Compare actual labels with predictions. Binary F1 uses class 1; macro-F1 averages class F1 values equally.
changed_f1 = f1_score(actual, changed_pred, zero_division=0)
comparison = pd.DataFrame()
comparison['Case'] = ['Completed example', 'Your variant']
comparison['Accuracy'] = [reference_accuracy, changed_accuracy]
comparison['Precision (class 1)'] = [reference_precision, changed_precision]
comparison['Recall (class 1)'] = [reference_recall, changed_recall]
comparison['F1 (class 1)'] = [reference_f1, changed_f1]
# Count errors explicitly so you do not need a matrix for every experiment.
reference_matrix = confusion_matrix(actual, reference_pred, labels=[0, 1])
# Count actual labels by row and predicted labels by column, in the specified label order.
changed_matrix = confusion_matrix(actual, changed_pred, labels=[0, 1])
comparison['False positives'] = [reference_matrix[0, 1], changed_matrix[0, 1]]
comparison['False negatives'] = [reference_matrix[1, 0], changed_matrix[1, 0]]
# Round only the displayed table; model selection still uses the unrounded scores.
display(comparison.round(3))
changed_prediction_count = np.count_nonzero(reference_pred != changed_pred)
print('Predictions that changed:', changed_prediction_count)
labels = [0, 1]
names = ['Class 0', 'Class 1']

#### Record your results

| Setting | Positive-class F1 | FP / FN | Brief observation |
|---|---|---|---|
| Completed example | ___ | ___ | ___ |
| My variant: ___ | ___ | ___ | ___ |

**Your observation:** Record FP/FN before and after the 20 label flips. Are all validation errors new? Explain why the original labels must be copied.

TODO: Write your response using the output.

<a id="module-c"></a>

## C — Gaussian Naive Bayes

GaussianNB models each feature with a class-specific Gaussian distribution and
assumes features are independent **conditional on class**. Low correlation in the pooled data does not establish independence within each class. A prior describes class probabilities before
observing the features; it is not the final prediction probability. Smoothing
adds a variance floor for numerical stability. It is not data normalization.

**Start here:** run C setup, then the one-feature warm-up below, then C2. C3 is an additional investigation; C1 variance smoothing is a technical extension. The module data already have within-class correlation; C3 adds an exact duplicate and further repeats evidence rather than introducing dependence for the first time.

In [ ]:
# PROVIDED SETUP — imports make tools available; they do not train a model.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.datasets import make_moons, make_classification
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
plt.rcParams['font.size'] = 11
plt.rcParams['figure.dpi'] = 100

# Make two illustrative Gaussian classes with different covariance patterns.
# These are invented unitless coordinates, not a manufacturing experiment.
rng = np.random.default_rng(19)
class_zero = rng.multivariate_normal([-1, 0], [[1, 0.7], [0.7, 1]], size=120)
class_one = rng.multivariate_normal([1, 0.6], [[1, -0.6], [-0.6, 1]], size=120)
X = np.vstack([class_zero, class_one])  # Stack the two class tables.
y = np.concatenate([np.zeros(120, dtype=int), np.ones(120, dtype=int)])
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)
print('Training / validation rows:', len(y_train), len(y_valid))

### Completed warm-up — one feature, two Gaussian models

Use only the first synthetic feature. For each class, GaussianNB learns a mean
and variance: the location and spread of a bell-shaped distribution. A
**likelihood** describes how compatible an input is with that class's distribution;
it is not the probability that the class is correct. The **prior** gives the class
probability before using the input. Combining prior and likelihood and normalizing
produces the model's posterior class probabilities.

Run this example unchanged. In the plot, locate each class mean and the overlap.
In the table, compare the class probabilities at inputs −1, 0 and 1. At every
input the two probabilities sum to 1. Overlap means the input need not identify
the class with certainty. These are model estimates, not verified physical risks.
The warm-up is explanatory, not a new exercise or a model-selection experiment.
C2 returns to the original two-feature inputs and changes only the prior.

In [ ]:
# PROVIDED WARM-UP — a separate model; preserve all module data and models.
# [:, [0]] keeps a two-dimensional table containing only the first feature.
intro_train = X_train[:, [0]]
intro_model = GaussianNB()
# Fit learns from the supplied training data. Keep validation/test data out of fitting.
intro_model.fit(intro_train, y_train)
intro_inputs = np.array([[-1.0], [0.0], [1.0]])  # Three illustrative new inputs.
# Return one probability per class for each input row; classes_ gives the column order.
intro_probabilities = intro_model.predict_proba(intro_inputs)
intro_table = pd.DataFrame()
intro_table['Feature 1'] = intro_inputs[:, 0]
intro_table['P(class 0 | input)'] = intro_probabilities[:, 0]
intro_table['P(class 1 | input)'] = intro_probabilities[:, 1]
# Round only the displayed table; model selection still uses the unrounded scores.
display(intro_table.round(3))
print('Class order:', intro_model.classes_)
print('Learned class priors:', intro_model.class_prior_)
# Read the learned mean and variance of feature 1 for each class.
mean_zero = intro_model.theta_[0, 0]
mean_one = intro_model.theta_[1, 0]
sd_zero = np.sqrt(intro_model.var_[0, 0])
sd_one = np.sqrt(intro_model.var_[1, 0])
intro_axis = np.linspace(X_train[:, 0].min() - 1, X_train[:, 0].max() + 1, 200)
# Supplied Gaussian density formula; no derivation or editing is needed.
density_zero = np.exp(-0.5 * ((intro_axis - mean_zero) / sd_zero) ** 2) / (sd_zero * np.sqrt(2 * np.pi))
density_one = np.exp(-0.5 * ((intro_axis - mean_one) / sd_one) ** 2) / (sd_one * np.sqrt(2 * np.pi))
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(intro_axis, density_zero, color='#2867A0', label='Class 0 fitted density')
ax.plot(intro_axis, density_one, color='#B63D42', label='Class 1 fitted density')
ax.axvline(mean_zero, color='#2867A0', linestyle='--', label='Class 0 mean')
ax.axvline(mean_one, color='#B63D42', linestyle='--', label='Class 1 mean')
ax.set_xlabel('Illustrative feature 1 (unitless)')
ax.set_ylabel('Probability density (not class probability)')
ax.set_title('One feature: learned location, spread and overlap')
ax.legend()
fig.tight_layout()
plt.show()

### C2 — Recommended: Change the prior

Use class priors [0.8, 0.2] instead of the balanced empirical prior. Do not change labels or threshold.

In [ ]:
# COMPLETED EXAMPLE — run unchanged.
actual = y_valid
reference_model = GaussianNB()  # Default empirical priors and small variance smoothing.
reference_model.fit(X_train, y_train)  # Learn class priors, means and variances.
# Use the fitted model to assign labels to these rows; predict does not train the model.
reference_pred = reference_model.predict(X_valid)

In [ ]:
# YOUR TURN — edit the marked setting/expression.
class_zero_prior = 0.5  # TODO: Try 0.8.
class_one_prior = 1 - class_zero_prior
changed_model = GaussianNB(priors=[class_zero_prior, class_one_prior])
# Fit learns from the supplied training data. Keep validation/test data out of fitting.
changed_model.fit(X_train, y_train)
# Use the fitted model to assign labels to these rows; predict does not train the model.
changed_pred = changed_model.predict(X_valid)

In [ ]:
# PROVIDED REPORTING — optional reading; run unchanged.
# compare the two predictions on the SAME validation labels.
# Class 1 is positive. zero_division=0 reports zero for an undefined metric.
reference_accuracy = accuracy_score(actual, reference_pred)
changed_accuracy = accuracy_score(actual, changed_pred)
reference_precision = precision_score(actual, reference_pred, zero_division=0)
changed_precision = precision_score(actual, changed_pred, zero_division=0)
reference_recall = recall_score(actual, reference_pred, zero_division=0)
changed_recall = recall_score(actual, changed_pred, zero_division=0)
# Compare actual labels with predictions. Binary F1 uses class 1; macro-F1 averages class F1 values equally.
reference_f1 = f1_score(actual, reference_pred, zero_division=0)
# Compare actual labels with predictions. Binary F1 uses class 1; macro-F1 averages class F1 values equally.
changed_f1 = f1_score(actual, changed_pred, zero_division=0)
comparison = pd.DataFrame()
comparison['Case'] = ['Completed example', 'Your variant']
comparison['Accuracy'] = [reference_accuracy, changed_accuracy]
comparison['Precision (class 1)'] = [reference_precision, changed_precision]
comparison['Recall (class 1)'] = [reference_recall, changed_recall]
comparison['F1 (class 1)'] = [reference_f1, changed_f1]
# Count errors explicitly so you do not need a matrix for every experiment.
reference_matrix = confusion_matrix(actual, reference_pred, labels=[0, 1])
# Count actual labels by row and predicted labels by column, in the specified label order.
changed_matrix = confusion_matrix(actual, changed_pred, labels=[0, 1])
comparison['False positives'] = [reference_matrix[0, 1], changed_matrix[0, 1]]
comparison['False negatives'] = [reference_matrix[1, 0], changed_matrix[1, 0]]
# Round only the displayed table; model selection still uses the unrounded scores.
display(comparison.round(3))
changed_prediction_count = np.count_nonzero(reference_pred != changed_pred)
print('Predictions that changed:', changed_prediction_count)
labels = [0, 1]
names = ['Class 0', 'Class 1']

#### Record your results

| Setting | Positive-class F1 | FP / FN | Brief observation |
|---|---|---|---|
| Completed example | ___ | ___ | ___ |
| My variant: ___ | ___ | ___ | ___ |

**Your observation:** After increasing the class-0 prior, record class-1 precision and recall. Which error becomes more common, and why is a prior different from a measured frequency in these unchanged data?

TODO: Write your response using the output.

### C3 — Additional: Duplicate evidence is not new information

Enable a duplicate of the first input feature. Observe labels AND predicted probabilities. The original inputs are already dependent within each class; the duplicate adds no new information.

In [ ]:
# COMPLETED EXAMPLE — run unchanged.
actual = y_valid
reference_model = GaussianNB()  # Default empirical priors and small variance smoothing.
reference_model.fit(X_train, y_train)  # Learn class priors, means and variances.
# Use the fitted model to assign labels to these rows; predict does not train the model.
reference_pred = reference_model.predict(X_valid)
# Return one probability per class for each input row; classes_ gives the column order.
reference_probability = reference_model.predict_proba(X_valid)[:, 1]

In [ ]:
# YOUR TURN — edit the marked setting/expression.
duplicate_feature = False  # TODO: Change to True.
variant_train = X_train.copy()
variant_valid = X_valid.copy()
if duplicate_feature:
    # column_stack appends an exact copy of feature 1 as a third column.
    variant_train = np.column_stack([X_train, X_train[:, 0]])
    variant_valid = np.column_stack([X_valid, X_valid[:, 0]])
changed_model = GaussianNB()
# Fit learns from the supplied training data. Keep validation/test data out of fitting.
changed_model.fit(variant_train, y_train)
# Use the fitted model to assign labels to these rows; predict does not train the model.
changed_pred = changed_model.predict(variant_valid)
# Return one probability per class for each input row; classes_ gives the column order.
changed_probability = changed_model.predict_proba(variant_valid)[:, 1]
print('First five original / duplicate probabilities:')
print(reference_probability[:5])
print(changed_probability[:5])

In [ ]:
# PROVIDED REPORTING — optional reading; run unchanged.
# compare the two predictions on the SAME validation labels.
# Class 1 is positive. zero_division=0 reports zero for an undefined metric.
reference_accuracy = accuracy_score(actual, reference_pred)
changed_accuracy = accuracy_score(actual, changed_pred)
reference_precision = precision_score(actual, reference_pred, zero_division=0)
changed_precision = precision_score(actual, changed_pred, zero_division=0)
reference_recall = recall_score(actual, reference_pred, zero_division=0)
changed_recall = recall_score(actual, changed_pred, zero_division=0)
# Compare actual labels with predictions. Binary F1 uses class 1; macro-F1 averages class F1 values equally.
reference_f1 = f1_score(actual, reference_pred, zero_division=0)
# Compare actual labels with predictions. Binary F1 uses class 1; macro-F1 averages class F1 values equally.
changed_f1 = f1_score(actual, changed_pred, zero_division=0)
comparison = pd.DataFrame()
comparison['Case'] = ['Completed example', 'Your variant']
comparison['Accuracy'] = [reference_accuracy, changed_accuracy]
comparison['Precision (class 1)'] = [reference_precision, changed_precision]
comparison['Recall (class 1)'] = [reference_recall, changed_recall]
comparison['F1 (class 1)'] = [reference_f1, changed_f1]
# Count errors explicitly so you do not need a matrix for every experiment.
reference_matrix = confusion_matrix(actual, reference_pred, labels=[0, 1])
# Count actual labels by row and predicted labels by column, in the specified label order.
changed_matrix = confusion_matrix(actual, changed_pred, labels=[0, 1])
comparison['False positives'] = [reference_matrix[0, 1], changed_matrix[0, 1]]
comparison['False negatives'] = [reference_matrix[1, 0], changed_matrix[1, 0]]
# Round only the displayed table; model selection still uses the unrounded scores.
display(comparison.round(3))
changed_prediction_count = np.count_nonzero(reference_pred != changed_pred)
print('Predictions that changed:', changed_prediction_count)
labels = [0, 1]
names = ['Class 0', 'Class 1']

#### Record your results

| Setting | Positive-class F1 | FP / FN | Brief observation |
|---|---|---|---|
| Completed example | ___ | ___ | ___ |
| My variant: ___ | ___ | ___ | ___ |

**Your observation:** Compare one displayed probability before/after duplication and the F1 scores. What new physical information did the duplicate provide?

TODO: Write your response using the output.

### C1 — Additional: Variance smoothing

Compare default var_smoothing=1e-9 with 0.1; then try 1.0. Keep data fixed.

In [ ]:
# COMPLETED EXAMPLE — run unchanged.
actual = y_valid
reference_model = GaussianNB()  # Default empirical priors and small variance smoothing.
reference_model.fit(X_train, y_train)  # Learn class priors, means and variances.
# Use the fitted model to assign labels to these rows; predict does not train the model.
reference_pred = reference_model.predict(X_valid)

In [ ]:
# YOUR TURN — edit the marked setting/expression.
smoothing = 1e-9  # TODO: Try 0.1.
changed_model = GaussianNB(var_smoothing=smoothing)
# Fit learns from the supplied training data. Keep validation/test data out of fitting.
changed_model.fit(X_train, y_train)
# Use the fitted model to assign labels to these rows; predict does not train the model.
changed_pred = changed_model.predict(X_valid)

In [ ]:
# PROVIDED REPORTING — optional reading; run unchanged.
# compare the two predictions on the SAME validation labels.
# Class 1 is positive. zero_division=0 reports zero for an undefined metric.
reference_accuracy = accuracy_score(actual, reference_pred)
changed_accuracy = accuracy_score(actual, changed_pred)
reference_precision = precision_score(actual, reference_pred, zero_division=0)
changed_precision = precision_score(actual, changed_pred, zero_division=0)
reference_recall = recall_score(actual, reference_pred, zero_division=0)
changed_recall = recall_score(actual, changed_pred, zero_division=0)
# Compare actual labels with predictions. Binary F1 uses class 1; macro-F1 averages class F1 values equally.
reference_f1 = f1_score(actual, reference_pred, zero_division=0)
# Compare actual labels with predictions. Binary F1 uses class 1; macro-F1 averages class F1 values equally.
changed_f1 = f1_score(actual, changed_pred, zero_division=0)
comparison = pd.DataFrame()
comparison['Case'] = ['Completed example', 'Your variant']
comparison['Accuracy'] = [reference_accuracy, changed_accuracy]
comparison['Precision (class 1)'] = [reference_precision, changed_precision]
comparison['Recall (class 1)'] = [reference_recall, changed_recall]
comparison['F1 (class 1)'] = [reference_f1, changed_f1]
# Count errors explicitly so you do not need a matrix for every experiment.
reference_matrix = confusion_matrix(actual, reference_pred, labels=[0, 1])
# Count actual labels by row and predicted labels by column, in the specified label order.
changed_matrix = confusion_matrix(actual, changed_pred, labels=[0, 1])
comparison['False positives'] = [reference_matrix[0, 1], changed_matrix[0, 1]]
comparison['False negatives'] = [reference_matrix[1, 0], changed_matrix[1, 0]]
# Round only the displayed table; model selection still uses the unrounded scores.
display(comparison.round(3))
changed_prediction_count = np.count_nonzero(reference_pred != changed_pred)
print('Predictions that changed:', changed_prediction_count)
labels = [0, 1]
names = ['Class 0', 'Class 1']

#### Record your results

| Setting | Positive-class F1 | FP / FN | Brief observation |
|---|---|---|---|
| smoothing=1e-9 (reference) | ___ | ___ | ___ |
| smoothing=0.1 | ___ | ___ | ___ |
| smoothing=1.0 | ___ | ___ | ___ |

**Your observation:** Use your three recorded F1 scores to describe the effect of smoothing. If labels do not change, does that prove the estimated probabilities are identical? Explain what the displayed label-based metrics can establish.

TODO: Write a short response using the output.

<a id="module-d"></a>

## D — Apply the models to PHM tool wear

The course feature table derives from the PHM Society 2010 Data Challenge:
945 cuts, 315 each for c1/c4/c6. Wear is the mean of three flute measurements
in micrometers. This is a course-defined classification task, not the original
competition benchmark or an industrial replacement rule. Normal combines
Slight (≤75 µm) and Intermediate (>75 and <150 µm); Severe is ≥150 µm.

Use the 11 Lab 5 sensor summaries. Wear, cutter ID and cut number are not inputs.
Forces are in N, vibrations in g, AE-RMS in V; scaled inputs are unitless.
Features summarize full recordings, not an inferred engaged-cut interval.
Use c1/c4 only with the same Lab split. Adjacent cuts can be correlated across
train/validation; these scores do not demonstrate transfer to another cutter.
Do not use c6 to choose settings. Inspecting many variants can overfit validation.

[Dataset README](https://github.com/WSU-AI-in-ME/ai-in-me-1/blob/main/data/phm2010/features/README.md) ·
[Data dictionary](https://github.com/WSU-AI-in-ME/ai-in-me-1/blob/main/data/phm2010/features/DATA_DICTIONARY.md).
Follow their provenance and rights notes; the course license does not relicense
source data. No raw recordings are required.

In [ ]:
# PROVIDED SETUP — imports make tools available; they do not train a model.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.datasets import make_moons, make_classification
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
plt.rcParams['font.size'] = 11
plt.rcParams['figure.dpi'] = 100

# Prefer a local CSV beside the notebook; otherwise retrieve the public course file.
data_path = Path('phm2010_features.csv')
if data_path.exists():
    data = pd.read_csv(data_path)
else:
    data = pd.read_csv('https://raw.githubusercontent.com/WSU-AI-in-ME/ai-in-me-1/main/data/phm2010/features/phm2010_features.csv')
# Reject outdated labels without changing the CSV.
expected = pd.Series(0, index=data.index)
expected.loc[data['wear_mean_um'] > 75] = 1
expected.loc[data['wear_mean_um'] >= 150] = 2
if not np.array_equal(expected, data['wear_level']):
    raise ValueError('Download the current course CSV with 75/150 micrometer boundaries.')
# Use c1/c4 only. Do not fit, score or plot c6 in this Practice.
development_rows = data['cutter_id'].isin(['c1', 'c4'])
development = data.loc[development_rows].copy()
class_text = development['wear_level'].astype(str)
strata = development['cutter_id'] + '_' + class_text
train, valid = train_test_split(development, test_size=0.20, random_state=42, stratify=strata)
train = train.sort_index()
valid = valid.sort_index()
sensor_features = ['force_x_mean', 'force_x_sd', 'force_y_mean', 'force_y_sd',
    'force_z_mean', 'force_z_sd', 'vibration_x_sd', 'vibration_y_sd',
    'vibration_z_sd', 'ae_rms_mean', 'ae_rms_sd']
X_train = train[sensor_features]
X_valid = valid[sensor_features]
y_train_multi = train['wear_level']
y_valid_multi = valid['wear_level']
# Binary 1 means Severe wear; multiclass 1 means Intermediate-wear.
train_severe = y_train_multi == 2
valid_severe = y_valid_multi == 2
# Convert each Boolean decision into a class label: True becomes 1, False becomes 0.
y_train = train_severe.astype(int)
# Convert each Boolean decision into a class label: True becomes 1, False becomes 0.
y_valid = valid_severe.astype(int)
scaler = StandardScaler()
# Fit learns from the supplied training data. Keep validation/test data out of fitting.
scaler.fit(X_train)
# Reuse the training scaling parameters; do not learn new scaling from validation/test rows.
X_train_scaled = scaler.transform(X_train)
# Reuse the training scaling parameters; do not learn new scaling from validation/test rows.
X_valid_scaled = scaler.transform(X_valid)
print('Training / validation rows:', len(train), len(valid))
display(pd.crosstab(train['cutter_id'], train['wear_level']))
display(pd.crosstab(valid['cutter_id'], valid['wear_level']))

### D1 — Additional: Binary kNN on sensor summaries

Change k=5 to k=15. Use Severe F1, and identify one matrix entry that changes or stays unchanged.

In [ ]:
# COMPLETED EXAMPLE — run unchanged.
actual = y_valid
reference_model = KNeighborsClassifier(n_neighbors=5, weights='uniform')
reference_model.fit(X_train_scaled, y_train)  # Store training inputs and labels.
reference_pred = reference_model.predict(X_valid_scaled)  # Predict validation labels.

In [ ]:
# YOUR TURN — edit the marked setting/expression.
k = 5  # TODO: Try 15.
changed_model = KNeighborsClassifier(n_neighbors=k)
# Fit learns from the supplied training data. Keep validation/test data out of fitting.
changed_model.fit(X_train_scaled, y_train)
# Use the fitted model to assign labels to these rows; predict does not train the model.
changed_pred = changed_model.predict(X_valid_scaled)

In [ ]:
# PROVIDED REPORTING — optional reading; run unchanged.
# compare the two predictions on the SAME validation labels.
# Class 1 is positive. zero_division=0 reports zero for an undefined metric.
reference_accuracy = accuracy_score(actual, reference_pred)
changed_accuracy = accuracy_score(actual, changed_pred)
reference_precision = precision_score(actual, reference_pred, zero_division=0)
changed_precision = precision_score(actual, changed_pred, zero_division=0)
reference_recall = recall_score(actual, reference_pred, zero_division=0)
changed_recall = recall_score(actual, changed_pred, zero_division=0)
# Compare actual labels with predictions. Binary F1 uses class 1; macro-F1 averages class F1 values equally.
reference_f1 = f1_score(actual, reference_pred, zero_division=0)
# Compare actual labels with predictions. Binary F1 uses class 1; macro-F1 averages class F1 values equally.
changed_f1 = f1_score(actual, changed_pred, zero_division=0)
comparison = pd.DataFrame()
comparison['Case'] = ['Completed example', 'Your variant']
comparison['Accuracy'] = [reference_accuracy, changed_accuracy]
comparison['Precision (class 1)'] = [reference_precision, changed_precision]
comparison['Recall (class 1)'] = [reference_recall, changed_recall]
comparison['F1 (class 1)'] = [reference_f1, changed_f1]
# Count errors explicitly so you do not need a matrix for every experiment.
reference_matrix = confusion_matrix(actual, reference_pred, labels=[0, 1])
# Count actual labels by row and predicted labels by column, in the specified label order.
changed_matrix = confusion_matrix(actual, changed_pred, labels=[0, 1])
comparison['False positives'] = [reference_matrix[0, 1], changed_matrix[0, 1]]
comparison['False negatives'] = [reference_matrix[1, 0], changed_matrix[1, 0]]
# Round only the displayed table; model selection still uses the unrounded scores.
display(comparison.round(3))
changed_prediction_count = np.count_nonzero(reference_pred != changed_pred)
print('Predictions that changed:', changed_prediction_count)
labels = [0, 1]
names = ['Class 0', 'Class 1']
names = ['Normal wear', 'Severe wear']

#### Record your results

| Setting | Positive-class F1 | FP / FN | Brief observation |
|---|---|---|---|
| Completed example | ___ | ___ | ___ |
| My variant: ___ | ___ | ___ | ___ |

**Your observation:** Record Severe F1 and changed-prediction count for k=5 and 15. If they are identical, state that explicitly. Why does this not establish performance on c6?

TODO: Write your response using the output.

### D2 — Recommended: GaussianNB with three wear classes

Compare GaussianNB with kNN (k=5) using three-class targets and the same scaled inputs. Use macro-F1, not binary F1.

In [ ]:
# COMPLETED EXAMPLE — run unchanged.
actual = y_valid_multi
reference_model = GaussianNB()
# Fit learns from the supplied training data. Keep validation/test data out of fitting.
reference_model.fit(X_train_scaled, y_train_multi)
# Use the fitted model to assign labels to these rows; predict does not train the model.
reference_pred = reference_model.predict(X_valid_scaled)

In [ ]:
# YOUR TURN — edit the marked setting/expression.
# TODO: Replace GaussianNB() with KNeighborsClassifier(n_neighbors=5).
changed_model = GaussianNB()
changed_model.fit(X_train_scaled, y_train_multi)  # Three-class labels, not binary labels.
# Use the fitted model to assign labels to these rows; predict does not train the model.
changed_pred = changed_model.predict(X_valid_scaled)

In [ ]:
# PROVIDED REPORTING — optional reading; run unchanged.
# macro-F1 weights each class equally; weighted-F1 uses support.
reference_f1 = f1_score(actual, reference_pred, labels=[0, 1, 2], average='macro', zero_division=0)
# Compare actual labels with predictions. Binary F1 uses class 1; macro-F1 averages class F1 values equally.
changed_f1 = f1_score(actual, changed_pred, labels=[0, 1, 2], average='macro', zero_division=0)
# Compare actual labels with predictions. Binary F1 uses class 1; macro-F1 averages class F1 values equally.
reference_weighted = f1_score(actual, reference_pred, labels=[0, 1, 2], average='weighted', zero_division=0)
# Compare actual labels with predictions. Binary F1 uses class 1; macro-F1 averages class F1 values equally.
changed_weighted = f1_score(actual, changed_pred, labels=[0, 1, 2], average='weighted', zero_division=0)
comparison = pd.DataFrame()
comparison['Case'] = ['Completed example', 'Your variant']
comparison['Macro-F1'] = [reference_f1, changed_f1]
comparison['Weighted-F1'] = [reference_weighted, changed_weighted]
# Round only the displayed table; model selection still uses the unrounded scores.
display(comparison.round(3))
labels = [0, 1, 2]
names = ['Class 0', 'Class 1', 'Class 2']
print(classification_report(actual, changed_pred, labels=labels, digits=3, zero_division=0))
names = ['Slight-wear', 'Intermediate-wear', 'Severe-wear']
# fig is the full figure; axes[0] and axes[1] are its two panels.
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
# Rows are actual classes and columns are predictions. Keep the same color scale.
ConfusionMatrixDisplay.from_predictions(actual, reference_pred, labels=labels,
    display_labels=names, ax=axes[0], colorbar=False, cmap='Blues', xticks_rotation=20)
ConfusionMatrixDisplay.from_predictions(actual, changed_pred, labels=labels,
    display_labels=names, ax=axes[1], colorbar=False, cmap='Blues', xticks_rotation=20)
axes[0].images[0].set_clim(0, len(actual))
axes[1].images[0].set_clim(0, len(actual))
axes[0].set_title('Completed example')
axes[1].set_title('Your variant')
fig.tight_layout()  # Make room for labels.
plt.show()  # Display the figure.

#### Record your results

| Setting | Macro-F1 | Class error to inspect | Brief observation |
|---|---|---|---|
| Completed example | ___ | ___ | ___ |
| My variant: ___ | ___ | ___ | ___ |

**Your observation:** Record both macro-F1 scores. Identify the most common GaussianNB actual-to-predicted error and its count. What does the shared c1/c4 split allow you to compare, and what does it not establish?

TODO: Write your response using the output.

### D3 — Additional: Change a GaussianNB decision threshold

Hold one fitted binary GaussianNB fixed and change its Severe threshold from 0.50 to 0.25. Compare FP/FN.

In [ ]:
# COMPLETED EXAMPLE — run unchanged.
actual = y_valid
reference_model = GaussianNB()
# Fit learns from the supplied training data. Keep validation/test data out of fitting.
reference_model.fit(X_train_scaled, y_train)
print('Probability columns:', reference_model.classes_)  # Order is [0, 1].
# Return one probability per class for each input row; classes_ gives the column order.
probabilities = reference_model.predict_proba(X_valid_scaled)
severe_probability = probabilities[:, 1]  # All rows, second column.
# The comparison is True where the estimated positive-class probability reaches the cutoff.
reference_pred = (severe_probability >= 0.50).astype(int)

In [ ]:
# YOUR TURN — edit the marked setting/expression.
threshold = 0.50  # TODO: Try 0.25.
# The comparison is True where the estimated positive-class probability reaches the cutoff.
is_severe = severe_probability >= threshold
# Convert each Boolean decision into a class label: True becomes 1, False becomes 0.
changed_pred = is_severe.astype(int)

In [ ]:
# PROVIDED REPORTING — optional reading; run unchanged.
# compare the two predictions on the SAME validation labels.
# Class 1 is positive. zero_division=0 reports zero for an undefined metric.
reference_accuracy = accuracy_score(actual, reference_pred)
changed_accuracy = accuracy_score(actual, changed_pred)
reference_precision = precision_score(actual, reference_pred, zero_division=0)
changed_precision = precision_score(actual, changed_pred, zero_division=0)
reference_recall = recall_score(actual, reference_pred, zero_division=0)
changed_recall = recall_score(actual, changed_pred, zero_division=0)
# Compare actual labels with predictions. Binary F1 uses class 1; macro-F1 averages class F1 values equally.
reference_f1 = f1_score(actual, reference_pred, zero_division=0)
# Compare actual labels with predictions. Binary F1 uses class 1; macro-F1 averages class F1 values equally.
changed_f1 = f1_score(actual, changed_pred, zero_division=0)
comparison = pd.DataFrame()
comparison['Case'] = ['Completed example', 'Your variant']
comparison['Accuracy'] = [reference_accuracy, changed_accuracy]
comparison['Precision (class 1)'] = [reference_precision, changed_precision]
comparison['Recall (class 1)'] = [reference_recall, changed_recall]
comparison['F1 (class 1)'] = [reference_f1, changed_f1]
# Count errors explicitly so you do not need a matrix for every experiment.
reference_matrix = confusion_matrix(actual, reference_pred, labels=[0, 1])
# Count actual labels by row and predicted labels by column, in the specified label order.
changed_matrix = confusion_matrix(actual, changed_pred, labels=[0, 1])
comparison['False positives'] = [reference_matrix[0, 1], changed_matrix[0, 1]]
comparison['False negatives'] = [reference_matrix[1, 0], changed_matrix[1, 0]]
# Round only the displayed table; model selection still uses the unrounded scores.
display(comparison.round(3))
changed_prediction_count = np.count_nonzero(reference_pred != changed_pred)
print('Predictions that changed:', changed_prediction_count)
labels = [0, 1]
names = ['Class 0', 'Class 1']
names = ['Normal wear', 'Severe wear']
# Lowering from 0.50 to 0.25 changes only probabilities in this half-open interval.
above_lower = severe_probability >= 0.25
below_upper = severe_probability < 0.50
in_interval = above_lower & below_upper
print('Probabilities in [0.25, 0.50):', np.count_nonzero(in_interval))
print('Actual thresholds compared: 0.50 and', threshold)

#### Record your results

| Setting | Positive-class F1 | FP / FN | Brief observation |
|---|---|---|---|
| Completed example | ___ | ___ | ___ |
| My variant: ___ | ___ | ___ | ___ |

**Your observation:** How many probabilities lie in [0.25, 0.50)? Use that count to explain the changed-prediction count and FP/FN at the two thresholds.

TODO: Write your response using the output.

<a id="module-e"></a>

## E — Optional depth: LDA and QDA

Linear Discriminant Analysis (LDA) shares a within-class covariance matrix across
classes; Quadratic Discriminant Analysis (QDA) allows a separate matrix per class.
Both use Gaussian class-conditional models. QDA is more flexible but must estimate
more parameters. These classifiers do not require a prior dimensionality-reduction
step. Use the synthetic Gaussian setup; this section is optional.

In [ ]:
# PROVIDED SETUP — imports make tools available; they do not train a model.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.datasets import make_moons, make_classification
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
plt.rcParams['font.size'] = 11
plt.rcParams['figure.dpi'] = 100

# Make two illustrative Gaussian classes with different covariance patterns.
# These are invented unitless coordinates, not a manufacturing experiment.
rng = np.random.default_rng(19)
class_zero = rng.multivariate_normal([-1, 0], [[1, 0.7], [0.7, 1]], size=120)
class_one = rng.multivariate_normal([1, 0.6], [[1, -0.6], [-0.6, 1]], size=120)
X = np.vstack([class_zero, class_one])  # Stack the two class tables.
y = np.concatenate([np.zeros(120, dtype=int), np.ones(120, dtype=int)])
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)
print('Training / validation rows:', len(y_train), len(y_valid))

### E1 — Additional: Shared versus class-specific covariance

Replace LDA with QDA. The generated classes have different covariances, but a finite validation sample need not reward extra flexibility.

In [ ]:
# COMPLETED EXAMPLE — run unchanged.
actual = y_valid
reference_model = LinearDiscriminantAnalysis()
# Fit learns from the supplied training data. Keep validation/test data out of fitting.
reference_model.fit(X_train, y_train)
# Use the fitted model to assign labels to these rows; predict does not train the model.
reference_pred = reference_model.predict(X_valid)

In [ ]:
# YOUR TURN — edit the marked setting/expression.
# TODO: Use QuadraticDiscriminantAnalysis().
changed_model = LinearDiscriminantAnalysis()
# Fit learns from the supplied training data. Keep validation/test data out of fitting.
changed_model.fit(X_train, y_train)
# Use the fitted model to assign labels to these rows; predict does not train the model.
changed_pred = changed_model.predict(X_valid)

In [ ]:
# PROVIDED REPORTING — optional reading; run unchanged.
# compare the two predictions on the SAME validation labels.
# Class 1 is positive. zero_division=0 reports zero for an undefined metric.
reference_accuracy = accuracy_score(actual, reference_pred)
changed_accuracy = accuracy_score(actual, changed_pred)
reference_precision = precision_score(actual, reference_pred, zero_division=0)
changed_precision = precision_score(actual, changed_pred, zero_division=0)
reference_recall = recall_score(actual, reference_pred, zero_division=0)
changed_recall = recall_score(actual, changed_pred, zero_division=0)
# Compare actual labels with predictions. Binary F1 uses class 1; macro-F1 averages class F1 values equally.
reference_f1 = f1_score(actual, reference_pred, zero_division=0)
# Compare actual labels with predictions. Binary F1 uses class 1; macro-F1 averages class F1 values equally.
changed_f1 = f1_score(actual, changed_pred, zero_division=0)
comparison = pd.DataFrame()
comparison['Case'] = ['Completed example', 'Your variant']
comparison['Accuracy'] = [reference_accuracy, changed_accuracy]
comparison['Precision (class 1)'] = [reference_precision, changed_precision]
comparison['Recall (class 1)'] = [reference_recall, changed_recall]
comparison['F1 (class 1)'] = [reference_f1, changed_f1]
# Count errors explicitly so you do not need a matrix for every experiment.
reference_matrix = confusion_matrix(actual, reference_pred, labels=[0, 1])
# Count actual labels by row and predicted labels by column, in the specified label order.
changed_matrix = confusion_matrix(actual, changed_pred, labels=[0, 1])
comparison['False positives'] = [reference_matrix[0, 1], changed_matrix[0, 1]]
comparison['False negatives'] = [reference_matrix[1, 0], changed_matrix[1, 0]]
# Round only the displayed table; model selection still uses the unrounded scores.
display(comparison.round(3))
changed_prediction_count = np.count_nonzero(reference_pred != changed_pred)
print('Predictions that changed:', changed_prediction_count)
labels = [0, 1]
names = ['Class 0', 'Class 1']
# fig is the full figure; axes[0] and axes[1] are its two panels.
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
# Rows are actual classes and columns are predictions. Keep the same color scale.
ConfusionMatrixDisplay.from_predictions(actual, reference_pred, labels=labels,
    display_labels=names, ax=axes[0], colorbar=False, cmap='Blues', xticks_rotation=20)
ConfusionMatrixDisplay.from_predictions(actual, changed_pred, labels=labels,
    display_labels=names, ax=axes[1], colorbar=False, cmap='Blues', xticks_rotation=20)
axes[0].images[0].set_clim(0, len(actual))
axes[1].images[0].set_clim(0, len(actual))
axes[0].set_title('Completed example')
axes[1].set_title('Your variant')
fig.tight_layout()  # Make room for labels.
plt.show()  # Display the figure.

In [ ]:
# PROVIDED — visualize both fitted models in the same two-feature space.
# Background colors are predictions; points show actual VALIDATION labels.
plot_train = X_train
plot_valid = X_valid
# Set plot limits from training inputs; empty regions are extrapolation.
x_axis = np.linspace(plot_train[:, 0].min() - 0.5, plot_train[:, 0].max() + 0.5, 180)
y_axis = np.linspace(plot_train[:, 1].min() - 0.5, plot_train[:, 1].max() + 0.5, 180)
grid_x, grid_y = np.meshgrid(x_axis, y_axis)  # Create a rectangular grid.
grid_inputs = np.column_stack([grid_x.ravel(), grid_y.ravel()])  # One point per row.
# Use the fitted model to assign labels to these rows; predict does not train the model.
reference_grid = reference_model.predict(grid_inputs).reshape(grid_x.shape)
# Use the fitted model to assign labels to these rows; predict does not train the model.
changed_grid = changed_model.predict(grid_inputs).reshape(grid_x.shape)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].contourf(grid_x, grid_y, reference_grid, levels=[-0.5, 0.5, 1.5], colors=['#DCEAF5', '#F7DFDF'])
axes[0].scatter(plot_valid[:, 0], plot_valid[:, 1], c=y_valid, cmap='coolwarm', edgecolors='black', s=22)
axes[0].set_xlabel('Illustrative feature 1 (unitless)')
axes[0].set_ylabel('Illustrative feature 2 (unitless)')
axes[0].set_title('Completed example')
axes[1].contourf(grid_x, grid_y, changed_grid, levels=[-0.5, 0.5, 1.5], colors=['#DCEAF5', '#F7DFDF'])
axes[1].scatter(plot_valid[:, 0], plot_valid[:, 1], c=y_valid, cmap='coolwarm', edgecolors='black', s=22)
axes[1].set_xlabel('Illustrative feature 1 (unitless)')
axes[1].set_ylabel('Illustrative feature 2 (unitless)')
axes[1].set_title('Your variant')
fig.suptitle('Synthetic data: blue = class 0; red = class 1')
fig.tight_layout()
plt.show()

#### Record your results

| Setting | Positive-class F1 | FP / FN | Brief observation |
|---|---|---|---|
| Completed example | ___ | ___ | ___ |
| My variant: ___ | ___ | ___ | ___ |

**Your observation:** Compare the shapes and F1 scores. Can different boundaries produce the same score? Are empty regions of the plot supported by observed data?

TODO: Write your response using the output.

### E2 — Additional: QDA with fewer observations

Fit QDA using only the first 20 training rows; keep validation unchanged. Compare with QDA using all 180 training rows.

After recording the 20-row result, change `training_count` to 40 and rerun the action and reporting cells. Preserve both results in the table.

In [ ]:
# COMPLETED EXAMPLE — run unchanged.
actual = y_valid
reference_model = QuadraticDiscriminantAnalysis()
# Fit learns from the supplied training data. Keep validation/test data out of fitting.
reference_model.fit(X_train, y_train)
# Use the fitted model to assign labels to these rows; predict does not train the model.
reference_pred = reference_model.predict(X_valid)

In [ ]:
# YOUR TURN — edit the marked setting/expression.
training_count = 180  # TODO: Try 20, then 40; avoid tiny or single-class subsets.
small_X = X_train[:training_count]  # Select the first training_count rows.
small_y = y_train[:training_count]
print('Subset counts:', np.bincount(small_y))  # Check both classes remain represented.
changed_model = QuadraticDiscriminantAnalysis()
# Fit learns from the supplied training data. Keep validation/test data out of fitting.
changed_model.fit(small_X, small_y)
# Use the fitted model to assign labels to these rows; predict does not train the model.
changed_pred = changed_model.predict(X_valid)

In [ ]:
# PROVIDED REPORTING — optional reading; run unchanged.
# compare the two predictions on the SAME validation labels.
# Class 1 is positive. zero_division=0 reports zero for an undefined metric.
reference_accuracy = accuracy_score(actual, reference_pred)
changed_accuracy = accuracy_score(actual, changed_pred)
reference_precision = precision_score(actual, reference_pred, zero_division=0)
changed_precision = precision_score(actual, changed_pred, zero_division=0)
reference_recall = recall_score(actual, reference_pred, zero_division=0)
changed_recall = recall_score(actual, changed_pred, zero_division=0)
# Compare actual labels with predictions. Binary F1 uses class 1; macro-F1 averages class F1 values equally.
reference_f1 = f1_score(actual, reference_pred, zero_division=0)
# Compare actual labels with predictions. Binary F1 uses class 1; macro-F1 averages class F1 values equally.
changed_f1 = f1_score(actual, changed_pred, zero_division=0)
comparison = pd.DataFrame()
comparison['Case'] = ['Completed example', 'Your variant']
comparison['Accuracy'] = [reference_accuracy, changed_accuracy]
comparison['Precision (class 1)'] = [reference_precision, changed_precision]
comparison['Recall (class 1)'] = [reference_recall, changed_recall]
comparison['F1 (class 1)'] = [reference_f1, changed_f1]
# Count errors explicitly so you do not need a matrix for every experiment.
reference_matrix = confusion_matrix(actual, reference_pred, labels=[0, 1])
# Count actual labels by row and predicted labels by column, in the specified label order.
changed_matrix = confusion_matrix(actual, changed_pred, labels=[0, 1])
comparison['False positives'] = [reference_matrix[0, 1], changed_matrix[0, 1]]
comparison['False negatives'] = [reference_matrix[1, 0], changed_matrix[1, 0]]
# Round only the displayed table; model selection still uses the unrounded scores.
display(comparison.round(3))
changed_prediction_count = np.count_nonzero(reference_pred != changed_pred)
print('Predictions that changed:', changed_prediction_count)
labels = [0, 1]
names = ['Class 0', 'Class 1']

In [ ]:
# PROVIDED — visualize both fitted models in the same two-feature space.
# Background colors are predictions; points show actual VALIDATION labels.
plot_train = X_train
plot_valid = X_valid
# Set plot limits from training inputs; empty regions are extrapolation.
x_axis = np.linspace(plot_train[:, 0].min() - 0.5, plot_train[:, 0].max() + 0.5, 180)
y_axis = np.linspace(plot_train[:, 1].min() - 0.5, plot_train[:, 1].max() + 0.5, 180)
grid_x, grid_y = np.meshgrid(x_axis, y_axis)  # Create a rectangular grid.
grid_inputs = np.column_stack([grid_x.ravel(), grid_y.ravel()])  # One point per row.
# Use the fitted model to assign labels to these rows; predict does not train the model.
reference_grid = reference_model.predict(grid_inputs).reshape(grid_x.shape)
# Use the fitted model to assign labels to these rows; predict does not train the model.
changed_grid = changed_model.predict(grid_inputs).reshape(grid_x.shape)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].contourf(grid_x, grid_y, reference_grid, levels=[-0.5, 0.5, 1.5], colors=['#DCEAF5', '#F7DFDF'])
axes[0].scatter(plot_valid[:, 0], plot_valid[:, 1], c=y_valid, cmap='coolwarm', edgecolors='black', s=22)
axes[0].set_xlabel('Illustrative feature 1 (unitless)')
axes[0].set_ylabel('Illustrative feature 2 (unitless)')
axes[0].set_title('Completed example')
axes[1].contourf(grid_x, grid_y, changed_grid, levels=[-0.5, 0.5, 1.5], colors=['#DCEAF5', '#F7DFDF'])
axes[1].scatter(plot_valid[:, 0], plot_valid[:, 1], c=y_valid, cmap='coolwarm', edgecolors='black', s=22)
axes[1].set_xlabel('Illustrative feature 1 (unitless)')
axes[1].set_ylabel('Illustrative feature 2 (unitless)')
axes[1].set_title('Your variant')
fig.suptitle('Synthetic data: blue = class 0; red = class 1')
fig.tight_layout()
plt.show()

#### Record your results

| Setting | Positive-class F1 | FP / FN | Brief observation |
|---|---|---|---|
| 180 rows (reference) | ___ | ___ | ___ |
| 20 rows | ___ | ___ | ___ |
| 40 rows | ___ | ___ | ___ |

**Your observation:** Record the 20-, 40- and 180-row results. For 20 versus 180, compare precision and recall as well as F1. Why is one nested subset insufficient to infer the general effect of sample size?

TODO: Write your response using the output.

## Self-check and references

Restart and run the chosen module to check independence. The solution shows one possible route; a different setting can give a different valid result.

- PHM Society: [2010 Data Challenge](https://phmsociety.org/phm_competition/2010-phm-society-conference-data-challenge/).
- [Course feature provenance](https://github.com/WSU-AI-in-ME/ai-in-me-1/blob/main/data/phm2010/features/README.md).
- scikit-learn: [neighbors](https://scikit-learn.org/stable/modules/neighbors.html),
  [Naive Bayes](https://scikit-learn.org/stable/modules/naive_bayes.html),
  [LDA/QDA](https://scikit-learn.org/stable/modules/lda_qda.html),
  [evaluation](https://scikit-learn.org/stable/modules/model_evaluation.html).

Optional: [Lab 5 follow-up](practice05_lab_followup.ipynb) and [self-check](practice05_lab_followup_solution.ipynb) cover Logistic threshold 0.25, PHM SVM boundaries and label regrouping. These are extra activities, not additions to the recommended route.